============================================================
PYSPARK JOINS & OPTIMIZATION - Interview Q&A with Code
============================================================
Swiss Re Interview Prep: Senior Application Engineer
Topic: Join Strategies and Performance Tuning
============================================================

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, expr, rand, floor



ANSWER:

| Join Type       | Result                                           |
|-----------------|--------------------------------------------------|
| inner           | Only matching rows from both tables              |
| left (outer)    | All from left + matching from right (null if no match) |
| right (outer)   | All from right + matching from left              |
| full (outer)    | All from both (null where no match)              |
| left_semi       | Only left rows that have a match (no right cols) |
| left_anti       | Only left rows that have NO match                |
| cross           | Cartesian product (every row × every row)        |

INTERVIEW TIP: "I use left_semi when I just want to filter a table based on
existence in another table, like 'give me all claims that have a policyholder'.
It's faster than a regular join because it doesn't bring in extra columns."


In [ ]:
def join_types_demo():
    spark = SparkSession.builder.appName("JoinsDemo").master("local[*]").getOrCreate()
    
    claims = spark.createDataFrame([
        ("CL001", "P1", 1000),
        ("CL002", "P2", 2000),
        ("CL003", "P3", 3000),  # P3 has no policy
    ], ["claim_id", "policy_id", "amount"])
    
    policies = spark.createDataFrame([
        ("P1", "Alice"),
        ("P2", "Bob"),
        ("P4", "Charlie"),  # P4 has no claim
    ], ["policy_id", "name"])
    
    print("--- INNER JOIN (Only matches) ---")
    claims.join(policies, "policy_id", "inner").show()
    
    print("--- LEFT JOIN (All claims, policies if exist) ---")
    claims.join(policies, "policy_id", "left").show()
    
    print("--- LEFT SEMI (Claims that HAVE a policy, no policy columns) ---")
    claims.join(policies, "policy_id", "left_semi").show()
    
    print("--- LEFT ANTI (Claims that DON'T have a policy) ---")
    claims.join(policies, "policy_id", "left_anti").show()
    
    spark.stop()


In [ ]:
join_types_demo()


ANSWER:

Spark automatically chooses from these strategies:

1. BROADCAST HASH JOIN (BHJ)
   - Small table is sent to all executors
   - No shuffle for the large table
   - Best when: One table < 10MB (configurable)
   - Hint: broadcast(df)

2. SORT-MERGE JOIN (SMJ)
   - Both tables are sorted by join key, then merged
   - Requires shuffle for both tables
   - Best when: Both tables are large

3. SHUFFLE HASH JOIN (SHJ)
   - Partitions data by join key, builds hash map
   - Faster than SMJ for unsorted data
   - Best when: One table is moderately smaller

HOW TO CHECK:
df.explain() will show the join strategy.

INTERVIEW TIP: "For Claims joining with small dimension tables like Regions or
Claim Types, I always use broadcast(small_df) to avoid shuffling the large Claims table."


In [ ]:
def join_strategies_demo():
    spark = SparkSession.builder.appName("JoinStrategies").master("local[*]").getOrCreate()
    
    # Large table (simulated)
    large_df = spark.range(0, 100000).withColumn("key", (col("id") % 100).cast("string"))
    
    # Small table (dimension)
    small_df = spark.createDataFrame([(str(i), f"Label_{i}") for i in range(100)], ["key", "label"])
    
    print("--- Without Broadcast (Sort Merge Join) ---")
    result_no_broadcast = large_df.join(small_df, "key")
    result_no_broadcast.explain()
    
    print("\n--- With Broadcast (Broadcast Hash Join) ---")
    result_broadcast = large_df.join(broadcast(small_df), "key")
    result_broadcast.explain()
    
    spark.stop()


In [ ]:
join_strategies_demo()


ANSWER:

PROBLEM: One join key has disproportionately more data (e.g., 90% of claims are 'Type A').
This causes all that data to go to ONE executor, leading to OOM or timeout.

SOLUTION 1: SALTING (Manual)
- Add a random "salt" (0-9) to the skewed key
- Explode the small table to have all salt variants
- Join on the salted key

SOLUTION 2: ADAPTIVE QUERY EXECUTION (AQE) - Spark 3.0+
- spark.sql.adaptive.enabled = true
- spark.sql.adaptive.skewJoin.enabled = true
- Spark automatically detects and splits skewed partitions

INTERVIEW TIP: "In Databricks, AQE is enabled by default. But I understand
the salting technique for cases where AQE isn't available or doesn't help."


In [ ]:
def skew_handling_demo():
    spark = SparkSession.builder \
        .appName("SkewDemo") \
        .master("local[*]") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.skewJoin.enabled", "true") \
        .getOrCreate()
    
    # Skewed Data: "A" has 90% of records
    data = [("A", i) for i in range(90000)] + [("B", i) for i in range(10000)]
    skewed_df = spark.createDataFrame(data, ["key", "value"])
    
    dim_df = spark.createDataFrame([("A", "Info A"), ("B", "Info B")], ["key", "info"])
    
    SALT_BUCKETS = 10
    
    # Step 1: Salt the skewed (large) table
    salted_large = skewed_df.withColumn(
        "salt", floor(rand() * SALT_BUCKETS).cast("int")
    ).withColumn(
        "salted_key", expr("concat(key, '_', salt)")
    )
    
    # Step 2: Explode the small (dimension) table
    salted_small = dim_df.withColumn(
        "salt_array", expr(f"sequence(0, {SALT_BUCKETS - 1})")
    ).select(
        "key", "info",
        expr("explode(salt_array) as salt")
    ).withColumn(
        "salted_key", expr("concat(key, '_', salt)")
    )
    
    # Step 3: Join on the salted key
    result = salted_large.join(salted_small, "salted_key", "inner")
    
    print(f"Salted Join Result Count: {result.count()}")
    print("Skew is now distributed across multiple partitions!")
    
    spark.stop()


In [ ]:
skew_handling_demo()


ANSWER:

ISSUE 1: Joining on different data types
- Cause: df1.key (String) JOIN df2.key (Integer)
- Fix: Cast both to the same type BEFORE the join
  df1.withColumn("key", col("key").cast("string"))

ISSUE 2: Cartesian Product (Exploding data)
- Cause: Join condition always matches (or is missing)
- Fix: Double-check join condition, use DISTINCT if needed

ISSUE 3: Too Many Shuffle Partitions
- Cause: spark.sql.shuffle.partitions = 200 (default)
- Fix: For small data, reduce to 10-50. For large, increase.

ISSUE 4: Multiple Joins on the Same Key
- Cause: Joining A with B, then result with C, all on same key
- Fix: Persist the intermediate result to avoid re-computation

INTERVIEW TIP: "Before any join, I always check:
1. Are the key types the same?
2. Are there NULLs in the join key?
3. Is one table much smaller (use broadcast)?
4. Is there data skew (use salting or AQE)?"


In [ ]:
def join_best_practices_demo():
    spark = SparkSession.builder.appName("BestPractices").master("local[*]").getOrCreate()
    
    # Issue 1: Type Mismatch
    df1 = spark.createDataFrame([("1", "A"), ("2", "B")], ["key_str", "val1"])
    df2 = spark.createDataFrame([(1, "X"), (2, "Y")], ["key_int", "val2"])
    
    # BAD: This might cause implicit casting or fail silently
    # df1.join(df2, df1.key_str == df2.key_int)
    
    # GOOD: Explicit cast
    df1_casted = df1.withColumn("key", col("key_str").cast("int"))
    result = df1_casted.join(df2, df1_casted.key == df2.key_int)
    result.show()
    
    spark.stop()


In [ ]:
join_best_practices_demo()

In [ ]:
if __name__ == "__main__":
    print("=" * 60)
    print("Q1: Join Types Demo")
    print("=" * 60)
    join_types_demo()
    
    print("\n" + "=" * 60)
    print("Q2: Join Strategies Demo")
    print("=" * 60)
    join_strategies_demo()
    
    print("\n" + "=" * 60)
    print("Q3: Skew Handling Demo")
    print("=" * 60)
    skew_handling_demo()
    
    print("\n" + "=" * 60)
    print("Q4: Join Best Practices Demo")
    print("=" * 60)
    join_best_practices_demo()
